<a href="https://colab.research.google.com/github/wsamuelw/autokeras-text-classification/blob/main/Qwen3_ASR_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# BLOCK 1 — Install the qwen-asr package and supporting libraries
# This project leverages the Qwen Automatic Speech Recognition (ASR) model to convert spoken audio into accurate text transcriptions
# ================================================================

# Install the official ASR Python package that supports model loading.
!pip install -q qwen-asr

# Install audio processing and torch
!pip install -q torch torchaudio librosa soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.8/416.8 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 124.1 MB/s eta 0:00:00


In [ ]:
# ================================================================
# BLOCK 2 — Import modules and check GPU
# ================================================================

import torch
import librosa
import soundfile as sf

from qwen_asr import Qwen3ASRModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
# ================================================================
# BLOCK 3 — Load the Qwen3-ASR model for inference
# ================================================================

# You can choose the 0.6B (faster) or 1.7B (higher accuracy) checkpoint
model_name = "Qwen/Qwen3-ASR-0.6B"

# Load the model directly via the package
asr_model = Qwen3ASRModel.from_pretrained(
    model_name,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)

print("ASR model loaded:", model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


preprocessor_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

ASR model loaded: Qwen/Qwen3-ASR-0.6B


In [ ]:
# ================================================================
# BLOCK 4 — Upload and play the voice audio file
# ================================================================

from google.colab import files
from IPython.display import Audio, display

# Upload the file
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]

print("Uploaded audio:", audio_file)

# Play the uploaded audio in Colab
display(Audio(audio_file))

Saving 李佳芯首談分手真相----我主動提出過結婚.wav to 李佳芯首談分手真相----我主動提出過結婚.wav
Uploaded audio: 李佳芯首談分手真相----我主動提出過結婚.wav


In [ ]:
# ================================================================
# BLOCK 5 — Convert to mono 16kHz
# ================================================================

audio, sr = librosa.load(audio_file, sr=16000, mono=True)

clean_path = "clean_audio.wav"
sf.write(clean_path, audio, 16000)

print("Normalized audio saved as:", clean_path)

Normalized audio saved as: clean_audio.wav


In [ ]:
# ================================================================
# BLOCK 6 — Transcribe the audio to text
# ================================================================

results = asr_model.transcribe(
    audio=clean_path,
    language=None,       # Automatic language ID if None
    return_time_stamps=False
)

# The results is a list of transcription infos
transcript = results[0].text

print("Transcription output:")
print(transcript)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Transcription output:
驚嘅，我好想結婚嘅，係啦，喺呢段感情裏邊。即係雖然啲人會覺得，哈，你又攞時候，你工作誒呢幾年即係咁好咁樣。


In [ ]:
# ================================================================
# BLOCK 7 — Save reference text for voice cloning
# ================================================================

with open("reference_text.txt", "w") as f:
    f.write(transcript)

print("Reference text saved to reference_text.txt")